# SkillForge — Train `components.tflite` on Google Colab (T4 GPU)

This notebook trains a **YOLOv8n-pose** multi-component keypoint detection model on your Roboflow circuit dataset.
It detects **4 component classes** (LED, resistor, wire, arduino_header) with **2 keypoints each** (connection pins).

### Pipeline Steps:
1. **Check GPU Runtime** (Tesla T4)
2. **Install Ultralytics & Roboflow**
3. **Download Labeled Dataset** from Roboflow
4. **Patch data.yaml** with kpt_shape if needed
5. **Train YOLOv8n-pose** (150 epochs)
6. **Validate & Inspect Results**
7. **Export to int8 TFLite** (`components.tflite`)
8. **Download Model & Labelmap** for the mobile app

## 1. Verify GPU Allocation
Make sure your Colab runtime is set to **T4 GPU** (*Runtime* → *Change runtime type* → *T4 GPU*).

In [ ]:
!nvidia-smi

## 2. Install Ultralytics, Roboflow & Dependencies

In [ ]:
!pip install --upgrade ultralytics roboflow tensorflow opencv-python onnx pyyaml

## 3. Download Dataset from Roboflow
**⚠️ PASTE YOUR ROBOFLOW DOWNLOAD CODE BELOW** after annotating & generating a version.

It will look something like:
```python
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("your-workspace").project("skillforge-components")
version = project.version(1)
dataset = version.download("yolov8")
```

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="KrUxiAqCplSr35skpEoI")
project = rf.workspace("utkarsh-singh-ofhfi").project("skillforge-components")
version = project.version(1)
dataset = version.download("yolov8")
print("Dataset downloaded successfully at:", dataset.location)


## 4. Patch data.yaml with Keypoint Shape
Newer Ultralytics versions require `kpt_shape` inside `data.yaml`, **not** as a train argument.
Components use **2 keypoints** per instance `[2, 3]` (x, y, visibility).

In [ ]:
import os
import glob
import yaml

# 1. Locate data.yaml
yaml_files = glob.glob(f"{dataset.location}/**/data.yaml", recursive=True)
if not yaml_files:
    yaml_files = glob.glob("**/data.yaml", recursive=True)
if not yaml_files:
    raise FileNotFoundError("Could not find data.yaml! Verify dataset download.")

data_yaml_path = os.path.abspath(yaml_files[0])
dataset_root = os.path.dirname(data_yaml_path)
print(f"Dataset root: {dataset_root}")
print(f"data.yaml path: {data_yaml_path}")

# 2. Check and fix train/valid/test paths
train_path = os.path.join(dataset_root, "train", "images")
valid_path = os.path.join(dataset_root, "valid", "images")
test_path = os.path.join(dataset_root, "test", "images")

print(f"  train/images exists: {os.path.exists(train_path)}")
print(f"  valid/images exists: {os.path.exists(valid_path)}")
print(f"  test/images exists:  {os.path.exists(test_path)}")

with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["train"] = train_path
data_cfg["val"] = valid_path if os.path.exists(valid_path) else train_path
if "test" in data_cfg and not os.path.exists(test_path):
    del data_cfg["test"]

# 3. Patch kpt_shape [2, 3] (2 keypoints: pin1, pin2 with [x, y, vis])
data_cfg["kpt_shape"] = [2, 3]

with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("
--- Patched data.yaml ---")
with open(data_yaml_path, "r") as f:
    print(f.read())


## 5. Train YOLOv8n-pose for Component Detection
Trains for **150 epochs** with 4 classes and 2 keypoints each `[2, 3]`.
More epochs than board_pose because we have more classes and diversity.

In [ ]:
from ultralytics import YOLO

# Load lightweight YOLOv8n-pose pretrained weights
model = YOLO("yolov8n-pose.pt")

# Train on GPU — kpt_shape is in data.yaml, NOT passed as train argument
results = model.train(
    data=data_yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,
    device=0,
    project="skillforge",
    name="components_v1",
    exist_ok=True,
    plots=True,
    save=True
)
print("Training Complete!")


## 6. Evaluate Validation Metrics

In [ ]:
from IPython.display import Image, display

# Display training loss and mAP curves
results_img = 'skillforge/components_v1/results.png'
if os.path.exists(results_img):
    display(Image(filename=results_img))

# Display confusion matrix
cm_img = 'skillforge/components_v1/confusion_matrix.png'
if os.path.exists(cm_img):
    display(Image(filename=cm_img))

# Display sample validation predictions
val_batch_img = 'skillforge/components_v1/val_batch0_pred.jpg'
if os.path.exists(val_batch_img):
    display(Image(filename=val_batch_img))

## 7. Export to int8 Quantized TFLite
Converts trained PyTorch weights to mobile-ready `.tflite` with int8 quantization.

In [ ]:
import glob
from ultralytics import YOLO

# Auto-find best.pt weights reliably
best_pt_candidates = glob.glob("**/components_v1/**/best.pt", recursive=True) or glob.glob("**/best.pt", recursive=True)
if not best_pt_candidates:
    raise FileNotFoundError("Could not find best.pt! Ensure training finished successfully.")
best_weights = best_pt_candidates[-1]
print(f"Found best weights at: {best_weights}")

# Export to TFLite (int8 quantized)
export_model = YOLO(best_weights)
tflite_path = export_model.export(format="tflite", int8=True, imgsz=640)
print(f"TFLite Model Exported at: {tflite_path}")


## 8. Package and Download Model for Mobile App
Creates `components.tflite` and `components_labelmap.json` for the SkillForge app.

In [ ]:
import os
import glob
import json
import shutil
from google.colab import files

# 1. Create components_labelmap.json
labelmap = {
    "model_name": "components",
    "version": "1.0.0",
    "classes": ["led", "resistor", "wire", "arduino_header"],
    "keypoints": [
        {"id": 0, "name": "pin1", "description": "First connection pin/leg"},
        {"id": 1, "name": "pin2", "description": "Second connection pin/leg"}
    ]
}
with open("components_labelmap.json", "w") as f:
    json.dump(labelmap, f, indent=2)

# 2. Locate generated tflite file
tflite_candidates = glob.glob("**/*int8*.tflite", recursive=True) or \n                    glob.glob("**/*components*.tflite", recursive=True) or \n                    glob.glob("**/*best*.tflite", recursive=True) or \n                    glob.glob("**/*.tflite", recursive=True)

if not tflite_candidates:
    raise FileNotFoundError("No .tflite files found! Check export cell.")

selected_tflite = tflite_candidates[0]
shutil.copy(selected_tflite, "components.tflite")
print(f"Using TFLite file: {selected_tflite} -> components.tflite ({os.path.getsize("components.tflite") / 1024 / 1024:.2f} MB)")
print("Downloading components.tflite and components_labelmap.json...")
files.download("components.tflite")
files.download("components_labelmap.json")
